# Stage 5 - Fairness and Bias Audit
This notebook computes group fairness metrics at the Stage 4 max-F1 threshold and documents disparity interpretation.

In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score

BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'


## Load Stage 4 Outputs

In [2]:
with open(PROCESSED_DIR / 'stage4_outputs.pkl', 'rb') as f:
    stage4 = pickle.load(f)
best_model = stage4['best_model']
X_test_eng = stage4['X_test_eng']
y_test = stage4['y_test']
borrower_type_test = stage4['borrower_type_test']
max_f1_threshold = stage4['max_f1_threshold']
print('Loaded test shape:', X_test_eng.shape)
print('Best model:', stage4['best_model_name'])


Loaded test shape: (2000, 49)
Best model: lr_l1


## Compute Group-wise Fairness Metrics
Use predictions at max-F1 threshold to calculate predicted positive rate, TPR, FPR, and precision for each borrower type.

In [3]:
y_prob_best = best_model.predict_proba(X_test_eng)[:, 1]
y_pred_best = (y_prob_best >= max_f1_threshold).astype(int)

rows = []
for group in sorted(borrower_type_test.unique()):
    mask = borrower_type_test == group
    y_true_g = y_test[mask]
    y_pred_g = y_pred_best[mask]
    tn, fp, fn, tp = confusion_matrix(y_true_g, y_pred_g).ravel()
    ppr = y_pred_g.mean()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    prec = precision_score(y_true_g, y_pred_g, zero_division=0)
    rows.append({'borrower_type': group, 'predicted_positive_rate': ppr, 'tpr_recall': tpr, 'fpr': fpr, 'precision': prec})

fairness_table = pd.DataFrame(rows)
print(fairness_table)


  borrower_type  predicted_positive_rate  tpr_recall       fpr  precision
0           gig                 0.091365    0.563636  0.032731   0.681319
1       migrant                 0.089157    0.656250  0.041775   0.567568
2         rural                 0.283531    0.727848  0.120650   0.688623


## Compute Disparity Ratios
For each fairness metric, compute highest/lowest ratio and flag values above 1.25.

In [4]:
disparity_rows = []
for metric in ['predicted_positive_rate', 'tpr_recall', 'fpr', 'precision']:
    values = fairness_table[metric].dropna()
    ratio = float(values.max() / values.min()) if values.min() > 0 else np.inf
    disparity_rows.append({'metric': metric, 'high_low_ratio': ratio, 'flag_ratio_gt_1_25': ratio > 1.25})
disparity_table = pd.DataFrame(disparity_rows)
print(disparity_table)


                    metric  high_low_ratio  flag_ratio_gt_1_25
0  predicted_positive_rate        3.180150                True
1               tpr_recall        1.291343                True
2                      fpr        3.686055                True
3                precision        1.213288               False


## Fairness Interpretation
Interpret whether disparity patterns are likely data-driven versus possible model over-penalisation.

In [5]:
print('Interpretation:')
print('- Compare fairness ratios with known synthetic-data default differences across borrower groups.')
print('- If PPR/TPR/FPR disparities exceed 1.25 materially, this suggests meaningful group performance divergence.')
print('- Divergence may partly reflect true base-rate differences, but elevated FPR for one group indicates possible over-penalisation risk that should be monitored in policy.')
print('- This stage documents disparities only; no mitigation is applied by design.')


Interpretation:
- Compare fairness ratios with known synthetic-data default differences across borrower groups.
- If PPR/TPR/FPR disparities exceed 1.25 materially, this suggests meaningful group performance divergence.
- Divergence may partly reflect true base-rate differences, but elevated FPR for one group indicates possible over-penalisation risk that should be monitored in policy.
- This stage documents disparities only; no mitigation is applied by design.


## Save Stage 5 Outputs

In [6]:
stage5_output = {
    **stage4,
    'y_pred_best': y_pred_best,
    'y_prob_best': y_prob_best,
    'fairness_table': fairness_table,
    'disparity_table': disparity_table,
}
out_path = PROCESSED_DIR / 'stage5_outputs.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(stage5_output, f)
print('Saved:', out_path)


Saved: d:\College\Sem-4\ML\Course Project\cursor-alt-cred-score\data\processed\stage5_outputs.pkl


## Stage 5 Summary

In [7]:
print('=' * 70)
print('STAGE 5 SUMMARY')
print('- Completed fairness audit at max-F1 decision threshold.')
print(f"- Fairness table shape: {fairness_table.shape}")
print(f"- Disparity table shape: {disparity_table.shape}")
print(f"- Metrics flagged (>1.25 ratio): {int(disparity_table['flag_ratio_gt_1_25'].sum())}")
print(f"- Saved outputs to: {out_path}")
print('=' * 70)


STAGE 5 SUMMARY
- Completed fairness audit at max-F1 decision threshold.
- Fairness table shape: (3, 5)
- Disparity table shape: (4, 3)
- Metrics flagged (>1.25 ratio): 3
- Saved outputs to: d:\College\Sem-4\ML\Course Project\cursor-alt-cred-score\data\processed\stage5_outputs.pkl
